# 2. Obtención de datos de restaurantes — API de Yelp

Los datos de restaurantes de Miami se obtienen de la **API de Yelp Fusion** (endpoint `GET /v3/businesses/search`), siguiendo el procedimiento del material del curso (`Código_API_Yelp.ipynb`).

- La API requiere autenticación con una **API Key** (header `Authorization: Bearer <KEY>`).
- ⚠️ La key **no se escribe en el notebook** porque este repositorio es público: se lee de la variable de entorno `YELP_API_KEY`.
- El resultado de la extracción está guardado en `Datasets/yelp_restaurants.csv`, por lo que el análisis es **reproducible sin credenciales**.

In [1]:
import os
from pathlib import Path
import requests
import pandas as pd

pd.set_option("display.max_columns", None)

# URL de la API (la key se obtiene creando una app en https://www.yelp.com/developers)
api_url = "https://api.yelp.com/v3/businesses/search"

# La key NO se escribe en el notebook (el repositorio es público). Se busca en:
#  1) la variable de entorno YELP_API_KEY
#  2) el archivo local yelp_api_key.txt (fuera del repositorio, no se sube a GitHub)
apikey = os.environ.get("YELP_API_KEY")
if not apikey:
    archivo_key = Path("../../../yelp_api_key.txt")
    if archivo_key.exists():
        apikey = archivo_key.read_text(encoding="utf-8").strip()

headers = {"Authorization": "Bearer %s" % apikey} if apikey else None

ciudad = "Miami"  # la misma ciudad usada para filtrar la base de clientes en 1_EDA
RUTA_CSV = "../../Datasets/yelp_restaurants.csv"

print("API Key configurada:", "Sí" if apikey else "No (se usará la extracción guardada)")

API Key configurada: Sí


## 2.1 Consulta a la API

Así se consulta la API para obtener restaurantes de la ciudad elegida. La respuesta es un JSON con tres claves: `businesses` (la lista de restaurantes), `total` (cuántos hay en la base de Yelp) y `region`.

In [2]:
if apikey:
    params = {"term": "restaurants", "location": ciudad, "limit": 50}
    response = requests.get(api_url, params=params, headers=headers, timeout=15)
    print("Status:", response.status_code)
    data = response.json()
    print("Claves de la respuesta:", list(data.keys()))
    print("En total la base de datos registra %d restaurants en %s" % (data["total"], ciudad))
    print("\nPrimer restaurante recibido:")
    b = data["businesses"][0]
    print({k: b.get(k) for k in ["id", "name", "rating", "review_count", "price"]})
else:
    print("Sin API Key: esta celda demuestra la consulta; el análisis continúa con la extracción guardada.")

Status: 200
Claves de la respuesta: ['businesses', 'total', 'region']
En total la base de datos registra 5900 restaurants en Miami

Primer restaurante recibido:
{'id': 'UXHxLN3DcDGI57uDIfCuJA', 'name': "Old's Havana Cuban Bar & Cocina", 'rating': 4.4, 'review_count': 3296, 'price': '$$'}


## 2.2 Extracción completa y aplanado a tabla

La API devuelve como máximo 50 resultados por llamada, así que se **pagina** con el parámetro `offset`. Cada negocio trae sus categorías como lista, por lo que al aplanar el JSON se genera **una fila por categoría** — la misma estructura del CSV provisto.

Para mantener el análisis reproducible, la re-extracción solo se ejecuta si se activa `REEXTRAER = True`; por defecto se carga `Datasets/yelp_restaurants.csv`.

In [3]:
REEXTRAER = False  # cambiar a True (con API Key configurada) para regenerar el CSV

def extraer_restaurantes_yelp(ciudad, paginas=4, por_pagina=50):
    """Descarga restaurantes desde la API de Yelp y devuelve un DataFrame (una fila por categoría)."""
    filas = []
    for i in range(paginas):
        params = {"term": "restaurants", "location": ciudad,
                  "limit": por_pagina, "offset": i * por_pagina}
        r = requests.get(api_url, params=params, headers=headers, timeout=15)
        r.raise_for_status()
        for b in r.json()["businesses"]:
            for cat in b.get("categories", [{}]):
                filas.append({
                    "alias": cat.get("alias"),
                    "title": cat.get("title"),
                    "id": b["id"],
                    "name": b["name"],
                    "price": b.get("price", "No especificado"),
                    "rating": b.get("rating"),
                    "review_count": b.get("review_count"),
                    "distance": b.get("distance"),
                    "coordinates_latitude": b["coordinates"]["latitude"],
                    "coordinates_longitude": b["coordinates"]["longitude"],
                    "location_address1": b["location"].get("address1"),
                    "city": b["location"].get("city"),
                })
    return pd.DataFrame(filas)

if REEXTRAER and apikey:
    yelp = extraer_restaurantes_yelp(ciudad)
    yelp.to_csv(RUTA_CSV, index=False, encoding="utf-8")
    print("Datos re-extraídos desde la API y guardados en", RUTA_CSV)
else:
    yelp = pd.read_csv(RUTA_CSV)
    print("Se cargó la extracción guardada:", RUTA_CSV)

print("Filas:", len(yelp))
yelp.head()

Se cargó la extracción guardada: ../../Datasets/yelp_restaurants.csv
Filas: 463


,alias,title,id,name,price,rating,review_count,distance,coordinates_latitude,coordinates_longitude,location_address1,city
0,asianfusion,Asian Fusion,K3ukx2e11xTRtYBU01dmrA,Salty Flame,No especificado,4.4,184,9568.050891,25.760220,-80.192670,1414 Brickell Ave,Miami
1,steak,Steakhouses,K3ukx2e11xTRtYBU01dmrA,Salty Flame,No especificado,4.4,184,9568.050891,25.760220,-80.192670,1414 Brickell Ave,Miami
2,cocktailbars,Cocktail Bars,K3ukx2e11xTRtYBU01dmrA,Salty Flame,No especificado,4.4,184,9568.050891,25.760220,-80.192670,1414 Brickell Ave,Miami
3,cuban,Cuban,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,Medio,4.4,2965,7035.905420,25.765594,-80.219238,1442 SW 8th St,Miami
4,bars,Bars,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,Medio,4.4,2965,7035.905420,25.765594,-80.219238,1442 SW 8th St,Miami


## 2.3 Estructura y deduplicación

Como cada restaurante aparece una vez por categoría, para analizar restaurantes se deduplica por `id`.

In [4]:
print("Filas en el dataset:", len(yelp))
print("Restaurantes únicos:", yelp["id"].nunique())
print("Ciudades:", yelp["city"].unique())
print("\nNulos por columna:")
print(yelp.isna().sum())

restaurantes = yelp.drop_duplicates(subset="id").copy()
print("\nDataset deduplicado:", restaurantes.shape)

Filas en el dataset: 463
Restaurantes únicos: 200
Ciudades: <StringArray>
['Miami']
Length: 1, dtype: str

Nulos por columna:
alias                    0
title                    0
id                       0
name                     0
price                    0
rating                   0
review_count             0
distance                 0
coordinates_latitude     0
coordinates_longitude    0
location_address1        0
city                     0
dtype: int64

Dataset deduplicado: (200, 12)


## 2.4 Análisis de los restaurantes obtenidos

In [5]:
print("=== Calificaciones (rating) ===")
print(restaurantes["rating"].describe().round(2))

print("\n=== Nivel de precio ===")
print(restaurantes["price"].value_counts())

print("\n=== Top 10 categorías gastronómicas (sobre todas las filas) ===")
print(yelp["title"].value_counts().head(10))

print("\n=== Top 10 restaurantes por cantidad de reseñas ===")
print(restaurantes.nlargest(10, "review_count")[["name", "rating", "review_count", "price"]].to_string(index=False))

=== Calificaciones (rating) ===
count    200.00
mean       4.34
std        0.34
min        2.70
25%        4.10
50%        4.40
75%        4.50
max        5.00
Name: rating, dtype: float64

=== Nivel de precio ===
price
Medio              95
No especificado    70
Alto               19
Bajo               12
Muy Alto            4
Name: count, dtype: int64

=== Top 10 categorías gastronómicas (sobre todas las filas) ===
title
Seafood               24
Cocktail Bars         22
New American          19
Cuban                 18
Mediterranean         17
Steakhouses           16
Italian               15
Sandwiches            14
Desserts              13
Breakfast & Brunch    13
Name: count, dtype: int64

=== Top 10 restaurantes por cantidad de reseñas ===
                              name  rating  review_count price
                        Versailles     4.0          6869 Medio
                       CVI.CHE 105     4.4          5946 Medio
   Old's Havana Cuban Bar & Cocina     4.4          296

## 2.5 Integración con el análisis de clientes de Miami

Se cruza la **oferta** (restaurantes de Yelp) con la **demanda** (clientes de Miami analizados en `1_EDA.ipynb`) en dos dimensiones:

1. **Preferencias alimenticias vs categorías gastronómicas**: ¿la oferta acompaña lo que los clientes prefieren comer?
2. **Estrato socioeconómico vs nivel de precio**: ¿hay restaurantes para cada capacidad de gasto? (ambas fuentes usan las mismas etiquetas: Bajo / Medio / Alto / Muy Alto)

In [6]:
# Demanda: clientes de Miami ya limpios (generados en 1_EDA.ipynb)
clientes = pd.read_csv("../../Datasets/clientes_miami_limpio.csv")
print("Clientes de Miami:", len(clientes), "| Restaurantes únicos:", len(restaurantes))

# Mapeo de preferencias alimenticias de los clientes a categorías (alias) de Yelp
MAPEO_PREFERENCIAS = {
    "Mariscos":    ["seafood", "sushi", "poke", "raw_food"],
    "Pescado":     ["seafood", "sushi", "poke", "fishnchips"],
    "Carnes":      ["steak", "bbq", "butcher", "argentine", "burgers"],
    "Vegetariano": ["vegetarian", "salad", "vegan"],
    "Vegano":      ["vegan"],
}

filas = []
for pref, aliases in MAPEO_PREFERENCIAS.items():
    ids_oferta = yelp.loc[yelp["alias"].isin(aliases), "id"].unique()
    oferta = restaurantes[restaurantes["id"].isin(ids_oferta)]
    filas.append({
        "preferencia": pref,
        "clientes": (clientes["preferencias_alimenticias"] == pref).sum(),
        "% clientes": round((clientes["preferencias_alimenticias"] == pref).mean() * 100, 1),
        "restaurantes": len(oferta),
        "% oferta": round(len(oferta) / len(restaurantes) * 100, 1),
        "rating medio": round(oferta["rating"].mean(), 2) if len(oferta) else None,
    })

demanda_oferta = pd.DataFrame(filas)
demanda_oferta

Clientes de Miami: 3186 | Restaurantes únicos: 200


,preferencia,clientes,% clientes,restaurantes,% oferta,rating medio
0,Mariscos,819,25.7,33,16.5,4.26
1,Pescado,307,9.6,33,16.5,4.26
2,Carnes,511,16.0,27,13.5,4.31
3,Vegetariano,728,22.8,9,4.5,4.59
4,Vegano,382,12.0,2,1.0,4.45


In [7]:
# Capacidad de gasto (estrato de los clientes) vs nivel de precio de los restaurantes
dist_clientes = (clientes["estrato_socioeconomico"].value_counts(normalize=True) * 100).round(1)
con_precio = restaurantes[restaurantes["price"] != "No especificado"]
dist_precio = (con_precio["price"].value_counts(normalize=True) * 100).round(1)

comparativa = pd.DataFrame({
    "% clientes (estrato)": dist_clientes,
    "% restaurantes (precio)": dist_precio,
}).reindex(["Bajo", "Medio", "Alto", "Muy Alto"])
print("Restaurantes con precio informado:", len(con_precio), "de", len(restaurantes))
comparativa

Restaurantes con precio informado: 130 de 200


,% clientes (estrato),% restaurantes (precio)
Bajo,14.0,9.2
Medio,24.6,73.1
Alto,36.4,14.6
Muy Alto,25.1,3.1


In [8]:
# Restaurantes recomendables por perfil de cliente (insumo para Recomendaciones.pdf)
print("=== Para el segmento VIP (estratos Alto/Muy Alto): rating >= 4.5 y precio Alto/Muy Alto ===")
vip = restaurantes[(restaurantes["rating"] >= 4.5) & (restaurantes["price"].isin(["Alto", "Muy Alto"]))]
print(vip.nlargest(5, "review_count")[["name", "rating", "review_count", "price"]].to_string(index=False))

print("\n=== Para los segmentos de menor gasto: rating >= 4.5 y precio Bajo/Medio ===")
eco = restaurantes[(restaurantes["rating"] >= 4.5) & (restaurantes["price"].isin(["Bajo", "Medio"]))]
print(eco.nlargest(5, "review_count")[["name", "rating", "review_count", "price"]].to_string(index=False))

=== Para el segmento VIP (estratos Alto/Muy Alto): rating >= 4.5 y precio Alto/Muy Alto ===
                         name  rating  review_count    price
                   Fratellino     4.8          1804     Alto
                      Bunbury     4.6           948     Alto
                   COTE Miami     4.5           902 Muy Alto
                 Luca Osteria     4.5           430     Alto
Manny's Wood Grill Steakhouse     4.6           280     Alto

=== Para los segmentos de menor gasto: rating >= 4.5 y precio Bajo/Medio ===
           name  rating  review_count price
  Bombay Darbar     4.5          1673 Medio
 The Taco Stand     4.5          1634 Medio
       Sanguich     4.5          1410 Medio
Doce Provisions     4.5          1023 Medio
     Sweet Dogs     4.7           755  Bajo


## Conclusiones

**Sobre la extracción:**
- La base de Yelp registra **más de 5.000 restaurantes** en Miami; la extracción trabajada cubre los **200 primeros del ranking de relevancia** (463 filas al expandir categorías).
- Las calificaciones están concentradas en valores altos: media **4.34**, rango 2.7–5.0 — el sesgo típico de Yelp, donde los locales mal calificados tienden a salir de los listados.

**Sobre la integración oferta–demanda:**
- **Mariscos** (la preferencia dominante: 25,7% de los clientes) está bien servida: 16,5% de los restaurantes, con buen rating (4.26). Lo mismo **Carnes** (16% vs 13,5%).
- La **brecha está en lo vegetariano/vegano**: 22,8% de los clientes es vegetariano y 12% vegano, pero solo el 4,5% y el 1% de los restaurantes atiende esas categorías — y los pocos que existen tienen los mejores ratings (4.59 y 4.45). Es la oportunidad de mercado más clara del análisis.
- **Desajuste de precios**: el 61,5% de los clientes es de estrato Alto/Muy Alto, pero solo el 17,7% de los restaurantes (con precio informado) es de precio Alto/Muy Alto; el 73,1% se concentra en precio Medio. Hay más capacidad de gasto que oferta premium.
- Se identificaron restaurantes concretos por perfil: **Fratellino, Bunbury y COTE Miami** (rating ≥ 4.5, precio alto) para el segmento VIP; **Bombay Darbar, The Taco Stand y Sanguich** (rating ≥ 4.5, precio medio/bajo) para los segmentos de menor gasto. Estos nombres alimentan las propuestas de `Recomendaciones.pdf`.